# MLP Bottleneck Analysis

Properties of MLP hidden layers: activation sparsity,
magnitude distributions, effective dimensionality,
and input-output bottleneck structure.

## Why This Matters

MLP bottleneck provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_bottleneck_analysis import (
    hidden_activation_sparsity, hidden_activation_magnitude,
    bottleneck_effective_dimension, input_output_bottleneck,
    mlp_bottleneck_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Hidden Activation Sparsity

High sparsity means few neurons fire — suggesting sparse, interpretable coding.

In [ ]:
for layer in range(4):
    result = hidden_activation_sparsity(model, tokens, layer=layer)
    flag = ' [SPARSE]' if result['is_sparse'] else ''
    print(f"  Layer {layer}: sparsity={result['mean_sparsity']:.4f}{flag}")
    for p in result['per_position']:
        bar = '█' * int(p['fraction_active'] * 20)
        print(f"    Pos {p['position']}: active={p['fraction_active']:.4f} {bar}")

## Hidden Activation Magnitude

How strongly do neurons fire when active?

In [ ]:
for layer in range(4):
    result = hidden_activation_magnitude(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_active={result['mean_active_magnitude']:.4f}, "
          f"max={result['max_activation']:.4f}, overall_mean={result['overall_mean']:.4f}")

## Bottleneck Effective Dimension

Even with high d_mlp, activations may live in a low-rank subspace.

In [ ]:
for layer in range(4):
    result = bottleneck_effective_dimension(model, tokens, layer=layer)
    print(f"  Layer {layer}: eff_rank={result['effective_rank']:.2f} / d_mlp={result['d_mlp']}, "
          f"dim_90%={result['dim_for_90_pct']}, compression={result['compression_ratio']:.4f}")

## Input-Output Bottleneck

Does the MLP compress then expand information?

In [ ]:
for layer in range(4):
    result = input_output_bottleneck(model, tokens, layer=layer)
    print(f"  Layer {layer}: pre={result['pre_rank']:.2f} → post={result['post_rank']:.2f} "
          f"→ out={result['output_rank']:.2f}  (bottleneck_ratio={result['bottleneck_ratio']:.4f})")

## Bottleneck Summary

In [ ]:
result = mlp_bottleneck_summary(model, tokens)
for p in result['per_layer']:
    sparse_flag = ' [SPARSE]' if p['is_sparse'] else ''
    print(f"  Layer {p['layer']}: sparsity={p['sparsity']:.4f}, "
          f"eff_rank={p['effective_rank']:.2f}, compression={p['compression_ratio']:.4f}{sparse_flag}")